## Imports


In [ ]:
import os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import csv
import json
import random
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from pprint import pprint

import numpy as np
import jax
import matplotlib.pyplot as plt
from tqdm.auto import trange

from pymoo.core.problem import ElementwiseProblem
from pymoo.core.sampling import Sampling
from pymoo.core.repair import Repair
from pymoo.core.variable import Real, Integer, Binary
from pymoo.core.mixed import MixedVariableMating, MixedVariableDuplicateElimination
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.mutation.pm import PM
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.optimize import minimize

from LINKS.Optimization import Tools, DifferentiableTools, MechanismRandomizer
from LINKS.Geometry import CurveEngine
from LINKS.CP import REFERENCE_POINTS, MAX_JOINTS, make_empty_submission, evaluate_submission


## Settings


In [ ]:
# Run this notebook inside the repository containing LINKS, utils, and the .npy files.
TARGET_PATH = Path('kangaroo_target_curves.npy')
TARGET_INDICES = (1,)              # 0: round; 1: no ears/tail; 2: full kangaroo
QUICK_CHECK = True                # Set False for the longer optimization.
RUN_SWEEP = False                 # True compares all joint counts and seeds below.
JOINT_COUNTS = (7, 9, 11)
SEEDS = (123, 456, 789)
COORDINATE_BOUNDS = (0.0, 5.0)
DISTANCE_WEIGHTS = (0.90, 0.95)

POPULATION_SIZE = 20 if QUICK_CHECK else 100
GA1_GENERATIONS = 5 if QUICK_CHECK else 200
GD_STEPS = 10 if QUICK_CHECK else 200
GA2_GENERATIONS = 5 if QUICK_CHECK else 100
MAX_GD_CANDIDATES = 4 if QUICK_CHECK else 12
GD_LEARNING_RATE = 4e-4
GD_MAX_BACKTRACKS = 12
GD_MAX_COORDINATE_STEP = 0.02
GD_MIN_IMPROVEMENT = 1e-10
EDGE_MUTATION_RATE = 0.02          # Per binary gene; keep structural changes infrequent.
MAX_SUBMITTED_DESIGNS = 1000
SHOW_FINAL_CURVE = True

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = Path('BC_DS_JC_advanced_results') / RUN_ID
ACTIVE_JOINT_COUNTS = JOINT_COUNTS if RUN_SWEEP else JOINT_COUNTS[:1]
ACTIVE_SEEDS = SEEDS if RUN_SWEEP else SEEDS[:1]

assert TARGET_PATH.is_file(), 'Open the notebook from the CP1 repository directory.'
assert all(0 <= i < 3 for i in TARGET_INDICES)
assert all(4 <= n <= MAX_JOINTS for n in ACTIVE_JOINT_COUNTS)
assert all(0 < w <= 1 for w in DISTANCE_WEIGHTS)
assert COORDINATE_BOUNDS[0] < COORDINATE_BOUNDS[1]
assert POPULATION_SIZE >= 4 and min(GA1_GENERATIONS, GA2_GENERATIONS, GD_STEPS) >= 1
assert 1 <= MAX_GD_CANDIDATES <= POPULATION_SIZE
assert 1 <= MAX_SUBMITTED_DESIGNS <= 1000
target_curves = np.load(TARGET_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print('Mode:', 'quick check' if QUICK_CHECK else 'longer search')
print('Targets:', TARGET_INDICES, '| Joints:', ACTIVE_JOINT_COUNTS, '| Seeds:', ACTIVE_SEEDS)
print('Results:', OUTPUT_DIR)


## Objective tools


In [ ]:
# Use the same physical scale and simulation resolution as the final evaluator.
compile_start = perf_counter()
PROBLEM_TOOLS = Tools(timesteps=200, max_size=MAX_JOINTS,
                      material=True, scaled=False, device='cpu')
PROBLEM_TOOLS.compile()
GRADIENT_TOOLS = DifferentiableTools(device='cpu')
GRADIENT_TOOLS.compile()
COMPILE_SECONDS = perf_counter() - compile_start
COUNTS = {'objective_calls': 0, 'gradient_calls': 0}

def objective_values(mech, target_curve):
    COUNTS['objective_calls'] += 1
    d, m = PROBLEM_TOOLS(
        mech['x0'], mech['edges'], mech['fixed_joints'], mech['motor'],
        target_curve, target_idx=mech['target_joint'],
    )
    return np.array([float(np.asarray(d)), float(np.asarray(m))])

def feasible(F, reference):
    return bool(np.all(np.isfinite(F)) and np.all(F >= 0) and np.all(F < reference))

def weighted_loss(F, reference, weight):
    return float(weight * F[0] / reference[0] + (1 - weight) * F[1] / reference[1])

def copy_mechanism(mech):
    return {k: v.copy() if isinstance(v, np.ndarray) else v for k, v in mech.items()}


## Mechanism encoding


In [ ]:
class MechanismProblem(ElementwiseProblem):
    def __init__(self, target_curve, n_joints, reference):
        self.N = n_joints
        self.target_curve = target_curve
        self.reference = np.asarray(reference, dtype=float)
        variables = {
            f'C{j}_{i}': Binary()
            for i in range(n_joints) for j in range(i) if (j, i) != (0, 1)
        }
        variables.update({f'X0{i}': Real(bounds=COORDINATE_BOUNDS)
                          for i in range(2 * n_joints)})
        # Motor base 0 is always fixed; crank joint 1 always moves.
        variables.update({f'fixed_nodes{i}': Binary() for i in range(2, n_joints)})
        variables['target'] = Integer(bounds=(1, n_joints - 1))
        super().__init__(vars=variables, n_obj=2, n_ieq_constr=2)

    def decode(self, genes):
        target = int(genes['target'])
        edges = [(0, 1)] + [
            (j, i) for i in range(self.N) for j in range(i)
            if (j, i) != (0, 1) and bool(genes[f'C{j}_{i}'])
        ]
        fixed = [0] + [i for i in range(2, self.N) if bool(genes[f'fixed_nodes{i}'])]
        return dict(
            x0=np.array([genes[f'X0{i}'] for i in range(2 * self.N)], dtype=float).reshape(self.N, 2),
            edges=np.asarray(edges, dtype=int).reshape(-1, 2),
            fixed_joints=np.asarray(fixed, dtype=int), motor=np.array([0, 1]),
            target_joint=target,
        )

    def encode(self, mech):
        x0 = np.asarray(mech['x0'], dtype=float)
        if x0.shape != (self.N, 2):
            raise ValueError('Use a separate problem for each joint count.')
        if not np.array_equal(np.asarray(mech['motor']).ravel(), [0, 1]):
            raise ValueError('This encoding requires motor indices [0, 1].')
        target = mech.get('target_joint')
        if target is None:
            target = mech.get('target_idx')
        target = self.N - 1 if target is None else int(target)
        edge_set = {tuple(sorted(map(int, e))) for e in mech['edges']}
        fixed_set = set(map(int, mech['fixed_joints']))
        if 0 not in fixed_set or 1 in fixed_set or target in fixed_set:
            raise ValueError('Motor base must be fixed; crank and tracing joint must move.')
        genes = {f'C{j}_{i}': (j, i) in edge_set
                 for i in range(self.N) for j in range(i) if (j, i) != (0, 1)}
        genes.update({f'X0{i}': float(v) for i, v in enumerate(x0.ravel())})
        genes.update({f'fixed_nodes{i}': i in fixed_set for i in range(2, self.N)})
        genes['target'] = target
        return genes

    def structurally_valid(self, mech):
        x0, fixed, target = mech['x0'], set(mech['fixed_joints']), mech['target_joint']
        lo, hi = COORDINATE_BOUNDS
        if (not np.all(np.isfinite(x0)) or np.any(x0 < lo) or np.any(x0 > hi)
                or not 1 <= target < self.N or target in fixed):
            return False
        if np.linalg.norm(x0[0] - x0[1]) < 1e-8:
            return False
        # All moving joints must connect to the motor component.
        reached = {0}
        for _ in range(self.N):
            for a, b in mech['edges']:
                if a in reached or b in reached:
                    reached.update((int(a), int(b)))
        return all(i in reached for i in range(self.N) if i not in fixed)

    def _evaluate(self, genes, out, *args, **kwargs):
        mech = self.decode(genes)
        F = objective_values(mech, self.target_curve) if self.structurally_valid(mech) else np.full(2, np.inf)
        # Invalid simulations receive a finite, infeasible penalty for NSGA-II.
        if not np.all(np.isfinite(F)) or np.any(F < 0):
            F = self.reference * 1e6
        out['F'] = F
        out['G'] = F / self.reference - (1 - 1e-8)

class OutputJointRepair(Repair):
    def _do(self, problem, X, **kwargs):
        repaired = []
        for source in X:
            genes = dict(source)
            target = int(np.clip(round(genes['target']), 1, problem.N - 1))
            moving = [1] + [i for i in range(2, problem.N) if not genes[f'fixed_nodes{i}']]
            genes['target'] = target if target in moving else moving[-1]
            repaired.append(genes)
        return np.asarray(repaired, dtype=object)


## Population helpers


In [ ]:
def gene_key(problem, genes):
    return tuple(genes[k] for k in problem.vars)

def unique_genes(problem, members):
    found, answer = set(), []
    for genes in members:
        key = gene_key(problem, genes)
        if key not in found:
            found.add(key)
            answer.append(dict(genes))
    return answer

def pareto_indices(F):
    # Two-objective minimization; duplicate objective pairs need only one entry.
    F = np.asarray(F)
    if not len(F):
        return np.array([], dtype=int)
    order = np.lexsort((F[:, 1], F[:, 0]))
    selected, best_material = [], np.inf
    for i in order:
        if F[i, 1] < best_material:
            selected.append(int(i))
            best_material = F[i, 1]
    return np.asarray(selected, dtype=int)

def spread_indices(indices, limit):
    indices = np.asarray(indices, dtype=int)
    if len(indices) <= limit:
        return indices
    return indices[np.linspace(0, len(indices) - 1, limit, dtype=int)]

def evaluate_members(problem, members, require_feasible=False):
    records = []
    for genes in unique_genes(problem, members):
        mech = problem.decode(genes)
        if not problem.structurally_valid(mech):
            continue
        F = objective_values(mech, problem.target_curve)
        if not np.all(np.isfinite(F)) or np.any(F < 0):
            continue
        if require_feasible and not feasible(F, problem.reference):
            continue
        records.append({'genes': genes, 'mech': mech, 'F': F})
    return records

def select_seed_genes(problem, records, limit):
    if not records:
        return []
    F = np.array([r['F'] for r in records])
    cv = np.maximum(F / problem.reference - (1 - 1e-8), 0).sum(axis=1)
    good = np.flatnonzero(cv == 0)
    front = good[pareto_indices(F[good])]
    chosen = list(spread_indices(front, limit))
    # Fill remaining slots with the next feasible tradeoffs, then least violations.
    ranking = np.lexsort((np.sum(F / problem.reference, axis=1), cv))
    selected = set(chosen)
    for i in ranking:
        if len(chosen) >= limit:
            break
        if int(i) not in selected:
            chosen.append(int(i))
            selected.add(int(i))
    return [dict(records[i]['genes']) for i in chosen]

def random_population(problem, seed):
    np.random.seed(seed)
    random.seed(seed)
    generator = MechanismRandomizer(min_size=min(6, problem.N),
                                   max_size=max(14, problem.N), device='cpu')
    members, seen = [], set()
    for _ in trange(20 * POPULATION_SIZE, desc=f'Initialize N={problem.N}', leave=False):
        mech = generator(n=problem.N)
        try:
            genes = problem.encode(mech)
        except ValueError:
            continue
        decoded = problem.decode(genes)
        if not problem.structurally_valid(decoded):
            continue
        F = objective_values(decoded, problem.target_curve)
        key = gene_key(problem, genes)
        if np.all(np.isfinite(F)) and np.all(F >= 0) and key not in seen:
            members.append(genes)
            seen.add(key)
        if len(members) == POPULATION_SIZE:
            break
    if len(members) < POPULATION_SIZE:
        raise RuntimeError(f'Only {len(members)} valid mechanisms found; check bounds and generator output.')
    return members


## Genetic search


In [ ]:
class SeedSampling(Sampling):
    def __init__(self, members):
        super().__init__()
        self.members = [dict(g) for g in members]

    def _do(self, problem, n_samples, **kwargs):
        if len(self.members) < n_samples:
            raise ValueError('The GA seed population is too small.')
        return np.asarray([dict(g) for g in self.members[:n_samples]], dtype=object)

def run_ga(problem, members, generations, seed):
    # Configure mutation inside MixedVariableMating, where it is actually applied.
    mutation = {
        Real: PM(prob=0.9, eta=20),
        Integer: PM(prob=0.3, eta=20, vtype=float, repair=RoundingRepair()),
        Binary: BitflipMutation(prob=1.0, prob_var=EDGE_MUTATION_RATE),
    }
    mating = MixedVariableMating(
        mutation=mutation, repair=OutputJointRepair(),
        eliminate_duplicates=MixedVariableDuplicateElimination(),
    )
    algorithm = NSGA2(
        pop_size=POPULATION_SIZE, sampling=SeedSampling(members), mating=mating,
        repair=OutputJointRepair(), eliminate_duplicates=MixedVariableDuplicateElimination(),
    )
    result = minimize(problem, algorithm, ('n_gen', generations), seed=seed,
                      verbose=False, save_history=False)
    return [dict(g) for g in result.pop.get('X')], int(result.algorithm.evaluator.n_eval)


## Combined gradient descent


In [ ]:
def refine_member(problem, genes, weight):
    current = problem.decode(genes)
    F = objective_values(current, problem.target_curve)
    history = []
    if not feasible(F, problem.reference):
        return dict(genes), history

    for step in range(GD_STEPS):
        COUNTS['gradient_calls'] += 1
        d, m, dg, mg = GRADIENT_TOOLS(
            [current['x0']], [current['edges']], [current['fixed_joints']],
            [current['motor']], problem.target_curve, [current['target_joint']],
        )
        # L = w * distance / distance_limit + (1-w) * material / material_limit.
        gradient = (weight * np.asarray(dg[0]) / problem.reference[0]
                    + (1 - weight) * np.asarray(mg[0]) / problem.reference[1])
        if not np.all(np.isfinite(gradient)):
            break                     # Preserve the last design validated by Tools.
        grad_size = float(np.max(np.abs(gradient)))
        if grad_size < 1e-12:
            break
        rate = min(GD_LEARNING_RATE, GD_MAX_COORDINATE_STEP / grad_size)
        old_loss = weighted_loss(F, problem.reference, weight)
        accepted = False

        for _ in range(GD_MAX_BACKTRACKS + 1):
            trial = copy_mechanism(current)
            trial['x0'] = np.clip(current['x0'] - rate * gradient, *COORDINATE_BOUNDS)
            if problem.structurally_valid(trial):
                trial_F = objective_values(trial, problem.target_curve)
                if feasible(trial_F, problem.reference):
                    trial_loss = weighted_loss(trial_F, problem.reference, weight)
                    if trial_loss < old_loss - GD_MIN_IMPROVEMENT:
                        current, F, accepted = trial, trial_F, True
                        history.append(dict(step=step + 1, weight=weight, loss=trial_loss,
                                            distance=float(F[0]), material=float(F[1]), rate=rate))
                        break
            rate *= 0.5
        if not accepted:
            break
    # Every returned update, including the final one, has passed Tools validation.
    return problem.encode(current), history

def refine_population(problem, members):
    records = evaluate_members(problem, members, require_feasible=True)
    if not records:
        print('No feasible GA1 candidates: skipping GD; GA2 will continue searching.')
        return [], []
    F = np.array([r['F'] for r in records])
    chosen = spread_indices(pareto_indices(F), MAX_GD_CANDIDATES)
    refined, logs = [], []
    for index in trange(len(chosen), desc='GD candidates', leave=False):
        record = records[int(chosen[index])]
        for weight in DISTANCE_WEIGHTS:
            # Each weight starts from the same original candidate.
            genes, history = refine_member(problem, record['genes'], weight)
            refined.append(genes)
            logs.append(dict(candidate=index, weight=weight, accepted_steps=len(history),
                             history=history))
    return unique_genes(problem, refined), logs


## Run GA → GD → GA


In [ ]:
all_records = {i: [] for i in TARGET_INDICES}
run_summaries = []

for target_index in TARGET_INDICES:
    target_curve = np.asarray(target_curves[target_index])
    reference = np.asarray(REFERENCE_POINTS[target_index], dtype=float)
    for n_joints in ACTIVE_JOINT_COUNTS:
        for seed in ACTIVE_SEEDS:
            label = f'target_{target_index + 1}_N{n_joints}_seed{seed}'
            print('\nStarting', label)
            problem = MechanismProblem(target_curve, n_joints, reference)
            before_counts = COUNTS.copy()
            started = perf_counter()
            initial = random_population(problem, seed)
            ga1, ga1_evals = run_ga(problem, initial, GA1_GENERATIONS, seed)
            ga1_records = evaluate_members(problem, initial + ga1, require_feasible=True)
            refined, gd_logs = refine_population(problem, ga1)

            # Keep original and refined designs; seed GA2 with diverse good candidates.
            seed_records = evaluate_members(problem, initial + ga1 + refined)
            ga2_seeds = select_seed_genes(problem, seed_records, POPULATION_SIZE)
            if len(ga2_seeds) < POPULATION_SIZE:
                raise RuntimeError('Too few valid GA2 seeds; inspect the first GA results.')
            ga2, ga2_evals = run_ga(problem, ga2_seeds, GA2_GENERATIONS, seed + 1)
            final_records = evaluate_members(problem, initial + ga1 + refined + ga2,
                                             require_feasible=True)
            all_records[target_index].extend(final_records)
            best_before = min((r['F'][0] for r in ga1_records), default=np.nan)
            best_after = min((r['F'][0] for r in final_records), default=np.nan)
            summary = dict(target=target_index + 1, joints=n_joints, seed=seed,
                           ga1_evaluations=ga1_evals, ga2_evaluations=ga2_evals,
                           objective_calls=COUNTS['objective_calls'] - before_counts['objective_calls'],
                           gradient_calls=COUNTS['gradient_calls'] - before_counts['gradient_calls'],
                           search_seconds=perf_counter() - started,
                           feasible_candidates=len(final_records),
                           best_distance_after_ga1=float(best_before),
                           best_distance_in_archive=float(best_after))
            run_summaries.append(summary)
            np.save(OUTPUT_DIR / f'{label}_candidates.npy',
                    np.asarray([r['mech'] for r in final_records], dtype=object))
            (OUTPUT_DIR / f'{label}_gd.json').write_text(json.dumps(gd_logs, indent=2))
            pprint(summary)


## Save submission


In [ ]:
submission = make_empty_submission()
selected_records = {}
for target_index, records in all_records.items():
    if not records:
        selected_records[target_index] = []
        print(f'Problem {target_index + 1}: no feasible designs found; increase the search budget.')
        continue
    # Pareto filtering retains both accurate and economical designs across all runs.
    F = np.asarray([r['F'] for r in records])
    keep = spread_indices(pareto_indices(F), MAX_SUBMITTED_DESIGNS)
    selected_records[target_index] = [records[int(i)] for i in keep]
    submission[f'Problem {target_index + 1}'] = [
        copy_mechanism(r['mech']) for r in selected_records[target_index]
    ]

submission_path = OUTPUT_DIR / 'ga_gd_ga_submission.npy'
np.save(submission_path, submission)
settings = {key: value for key, value in list(globals().items())
            if key.isupper() and isinstance(value, (str, bool, int, float, tuple))}
settings['compile_seconds'] = COMPILE_SECONDS
settings['timing_note'] = 'First-use JAX tracing may also be included in search_seconds.'
(OUTPUT_DIR / 'settings.json').write_text(json.dumps(settings, indent=2))
with (OUTPUT_DIR / 'run_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(run_summaries[0]))
    writer.writeheader()
    writer.writerows(run_summaries)
print('Saved:', submission_path)
print('Design counts:', {k: len(v) for k, v in submission.items()})


## Official evaluation


In [ ]:
# Score the actual saved submission using the repository evaluator.
scores = evaluate_submission(str(submission_path), target_curves=str(TARGET_PATH))
pprint(scores)
(OUTPUT_DIR / 'official_scores.txt').write_text(str(scores))
if len(TARGET_INDICES) < 3:
    print('Only selected targets were optimized. Unselected targets contribute zero to the overall score.')

# Score each experiment using the same evaluator for comparisons across seeds/joint counts.
for summary in run_summaries:
    label = f"target_{summary['target']}_N{summary['joints']}_seed{summary['seed']}"
    trial = make_empty_submission()
    candidate_file = OUTPUT_DIR / f'{label}_candidates.npy'
    trial[f"Problem {summary['target']}"] = list(np.load(candidate_file, allow_pickle=True))[:MAX_SUBMITTED_DESIGNS]
    trial_scores = evaluate_submission(trial, target_curves=str(TARGET_PATH))
    (OUTPUT_DIR / f'{label}_official_scores.txt').write_text(str(trial_scores))
    print(label)
    pprint(trial_scores)


## Final curve


In [ ]:
# Only optimized-result plots; disable with SHOW_FINAL_CURVE = False.
if SHOW_FINAL_CURVE:
    curve_engine = CurveEngine(resolution=PROBLEM_TOOLS.timesteps, normalize_scale=False)
    for target_index, records in selected_records.items():
        if not records:
            continue
        best = min(records, key=lambda r: r['F'][0])
        mech = best['mech']
        solutions = np.asarray(PROBLEM_TOOLS.solver(
            [mech['x0']], [mech['edges']], [mech['fixed_joints']], [mech['motor']],
        ))
        traced_curve = solutions[0, mech['target_joint']]
        print(f"Problem {target_index + 1}: distance={best['F'][0]:.6g}, material={best['F'][1]:.6g}")
        # Avoid JAX boolean-tracing errors in the visualization helper.
        with jax.disable_jit():
            curve_engine.visualize_comparison(traced_curve, np.asarray(target_curves[target_index]))
        plt.show()
